# cuDF Game Analytics Benchmark

This notebook uses the repository's pinned environment and shared JSON result format. It runs CPU and GPU measurements in separate processes with identical deterministic inputs; timing values are never entered by hand.

The pilot environment is CPython 3.12, NumPy 2.3.4, pandas 3.0.0, and cuDF 26.10.2 (CUDA 12). A compatible NVIDIA GPU and driver are required for the GPU run. See [`BENCHMARK_STANDARD.md`](https://github.com/jjfishjj/cudf-game-analytics-benchmark/blob/main/BENCHMARK_STANDARD.md) and [NVIDIA's installation guide](https://docs.rapids.ai/install/).

The default ref is `main` for use after the pull request is merged. To test an unmerged branch, set `BENCHMARK_REF` to that branch name. Each run records the exact Git commit in its JSON output.

In [ ]:
from pathlib import Path
import subprocess

BENCHMARK_REF = 'main'
REPO_URL = 'https://github.com/jjfishjj/cudf-game-analytics-benchmark.git'
REPO_DIR = Path('/content/cudf-game-analytics-benchmark')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BENCHMARK_REF], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BENCHMARK_REF, REPO_URL, str(REPO_DIR)], check=True)
%cd /content/cudf-game-analytics-benchmark
!python -m pip install -q uv
!uv sync --locked --extra gpu --extra dev

## Run matched CPU and GPU measurements

Use the same rows, seed, dimension-table size, warm-up, and repetitions for both backends. The setup/transfer phase is recorded separately; GPU work is synchronized before and after each timed operation. Output files conform to `benchmark-results.schema.json`.

In [ ]:
COMMON = ['--rows', '5000000', '--dimension-rows', '1000000', '--seed', '42', '--warmups', '1', '--repetitions', '5']
python = str(Path('.venv/bin/python'))
subprocess.run([python, 'benchmarks/run_benchmark.py', '--backend', 'cpu', *COMMON, '--output', 'results/colab-cpu.json'], check=True)
subprocess.run([python, 'benchmarks/run_benchmark.py', '--backend', 'gpu', *COMMON, '--output', 'results/colab-gpu.json'], check=True)
print('Saved results/colab-cpu.json and results/colab-gpu.json')

## Results and interpretation

Compare medians only when the CPU and GPU runs use matching inputs and the same machine/software environment. Preserve the JSON files with their raw samples and environment metadata. The notebook no longer fills missing timing values or charts from hard-coded historical numbers.